# Notebook 03 — Molecular Basis of Mutation

**Module:** 06 — Genetics and Evolution  
**Notebook:** 03 of 12  
**Prerequisites:** Module 05 NB03 (central dogma), NB04 (genome scale)  
**Time estimate:** 60 minutes

---
## Step 1 — Motivation

Every variant in a VCF file originated as a mutation. Understanding *how* mutations
arise — which DNA contexts are prone to which mutation types, how repair pathways fail,
what distinguishes germline from somatic mutations — is required to interpret mutation
signatures in cancer genomics and to understand why some genomic regions are more
variable than others.

---
## Step 3 — Biological Background

### Mutation Types

| Type | Mechanism | Example consequence |
|------|-----------|---------------------|
| Point mutation (SNP/SNV) | Replication error, deamination, UV | Synonymous, missense, nonsense |
| Transition | Purine↔purine or pyrimidine↔pyrimidine | More common than transversions |
| Transversion | Purine↔pyrimidine | Less common; detected by Ti/Tv ratio |
| Insertion/deletion | Replication slippage at repeats | Frameshift if not multiple of 3 |
| Copy number variant (CNV) | Non-allelic homologous recombination | Gene dosage change |
| Inversion | Double-strand break + inverted rejoining | May disrupt gene regulation |
| Translocation | Break + join of different chromosomes | Oncogenic fusions (BCR-ABL) |

### Mutation Mechanisms

**Deamination of cytosine:** C → U (reads as T) → CpG→TpG transitions are the most
common SNP class in the human genome. Methylated CpGs deaminate 10–50× faster.

**UV-induced dimers:** Adjacent pyrimidines (CC, TT) form covalent dimers under UV
light → characteristic C→T or CC→TT 'UV signature' seen in skin cancer.

**APOBEC activity:** A family of cytidine deaminases causes C→T and C→G mutations
in TC contexts — a major mutation signature in breast and cervical cancers.

### DNA Repair Pathways

| Pathway | What it fixes | Failure consequence |
|---------|-------------|--------------------|
| Base excision repair (BER) | Oxidised/deaminated bases | Accumulation of oxidative damage |
| Nucleotide excision repair (NER) | Bulky adducts, UV dimers | Xeroderma pigmentosum (XP) |
| Mismatch repair (MMR) | Replication errors | Lynch syndrome (colorectal cancer) |
| Homologous recombination (HR) | Double-strand breaks | BRCA1/2 mutations → breast cancer |

### Germline vs. Somatic Mutations

- **Germline:** present in every cell; inherited; detected by GWAS; typically rare
- **Somatic:** arise in a specific cell lineage during life; not inherited; cancer-driving;
  detected by comparing tumour vs. matched normal
- **De novo:** new germline mutation not present in either parent (rate ~40–70 per generation
  in humans; increases with paternal age)

---
## Step 4 — Mathematical Explanation

**Transition/transversion ratio (Ti/Tv):**
- Expected under random mutation: Ti/Tv ≈ 0.5 (2 transversions vs 1 transition per site)
- Observed in human genome: Ti/Tv ≈ 2.1 (transitions are favoured)
- In coding regions: Ti/Tv ≈ 3.0 (selection purifies transversions more strongly)
- GATK quality filter: Ti/Tv < 1.8 in WGS suggests excessive false positives

**Mutation-selection balance:** At equilibrium, the frequency of a deleterious allele
is: q* ≈ μ/s (additive) or q* ≈ √(μ/s) (recessive). Where μ is the mutation rate
and s is the selection coefficient.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
# Cell 6.1 — Ti/Tv ratio calculator
TRANSITIONS   = {('A','G'), ('G','A'), ('C','T'), ('T','C')}
TRANSVERSIONS = {('A','C'), ('C','A'), ('A','T'), ('T','A'),
                 ('G','C'), ('C','G'), ('G','T'), ('T','G')}

def titv_ratio(ref_alleles: list, alt_alleles: list) -> dict:
    """
    Compute Ti/Tv ratio from lists of reference and alternate alleles.
    Only considers single-nucleotide changes.
    """
    ti, tv = 0, 0
    for ref, alt in zip(ref_alleles, alt_alleles):
        ref, alt = ref.upper(), alt.upper()
        if len(ref) != 1 or len(alt) != 1 or ref == alt:
            continue  # skip indels and non-variant sites
        if (ref, alt) in TRANSITIONS:
            ti += 1
        elif (ref, alt) in TRANSVERSIONS:
            tv += 1
    return {'ti': ti, 'tv': tv, 'titv': ti / tv if tv > 0 else np.inf}

# Simulate a set of SNPs with realistic Ti/Tv
rng = np.random.default_rng(42)
bases = ['A', 'C', 'G', 'T']

# Weight transitions 2.1× more than transversions
def random_snp(ref, rng, ti_weight=2.1):
    ti_alts = [b for b in bases if b != ref and (ref, b) in TRANSITIONS]
    tv_alts = [b for b in bases if b != ref and (ref, b) in TRANSVERSIONS]
    candidates = ti_alts * int(ti_weight * 2) + tv_alts * 2
    return rng.choice(candidates)

n_snps = 5000
refs = rng.choice(bases, size=n_snps)
alts = [random_snp(r, rng) for r in refs]

result = titv_ratio(list(refs), alts)
print(f"Simulated {n_snps} SNPs:")
print(f"  Transitions:  {result['ti']}")
print(f"  Transversions:{result['tv']}")
print(f"  Ti/Tv ratio:  {result['titv']:.3f}")
print(f"  (Human WGS expected: ~2.1)")

In [ ]:
# Cell 6.2 — CpG mutation hotspot: deamination of methylated cytosine
def simulate_cpg_mutations(seq_length: int, n_generations: int,
                            mu_base: float = 1.2e-8,
                            cpg_multiplier: float = 10.0,
                            seed: int = 42) -> dict:
    """
    Simulate sequence evolution with CpG hotspot.
    Returns counts of CpG→TpG and non-CpG C→T transitions.
    """
    rng = np.random.default_rng(seed)
    # Simplified: track positions that are CpG vs. non-CpG
    # Assume 4% of positions are CpG (rough human estimate)
    n_cpg = int(seq_length * 0.04)
    n_other = seq_length - n_cpg

    # Expected mutations
    mu_cpg   = mu_base * cpg_multiplier * n_generations
    mu_other = mu_base * n_generations

    cpg_muts   = rng.binomial(n_cpg,   min(mu_cpg, 1))
    other_muts = rng.binomial(n_other, min(mu_other, 1))

    return {'cpg_mutations': cpg_muts, 'other_mutations': other_muts,
            'cpg_rate': cpg_muts / n_cpg, 'other_rate': other_muts / n_other,
            'relative_rate': (cpg_muts / n_cpg) / (other_muts / n_other)}

# Simulate 1 million bp over 100,000 generations (representing ~1000 years)
sim = simulate_cpg_mutations(1_000_000, 100_000)
print("CpG hotspot simulation (1 Mb, 100K generations):")
print(f"  CpG positions mutated: {sim['cpg_mutations']}  (rate={sim['cpg_rate']:.2e})")
print(f"  Other positions mutated: {sim['other_mutations']}  (rate={sim['other_rate']:.2e})")
print(f"  CpG relative rate: {sim['relative_rate']:.1f}× higher (expected ~10×)")

In [ ]:
# Cell 6.3 — Mutation-selection balance
def mutation_selection_equilibrium(mu: float, s: float, dominance: str = 'additive') -> float:
    """
    Equilibrium frequency of deleterious allele under mutation-selection balance.
    """
    if dominance == 'additive':
        return mu / s
    elif dominance == 'recessive':
        return np.sqrt(mu / s)
    else:
        raise ValueError(dominance)

print("Mutation-selection balance equilibrium frequencies:")
print(f"  {'s':>8} {'q* additive':>14} {'q* recessive':>14}")
mu = 1e-5   # per-gene mutation rate
for s in [0.001, 0.01, 0.1, 0.5, 1.0]:
    q_add = mutation_selection_equilibrium(mu, s, 'additive')
    q_rec = mutation_selection_equilibrium(mu, s, 'recessive')
    print(f"  {s:>8.3f} {q_add:>14.2e} {q_rec:>14.2e}")
print("\nRecessive deleterious alleles persist at much higher frequency")
print("because carriers (Aa) are not selected against.")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Mutation spectrum + equilibrium frequency
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: SBS1 (CpG deamination) vs. random mutation spectrum
ax = axes[0]
mut_types = ['C>T (CpG)', 'C>T (non-CpG)', 'C>A', 'C>G', 'T>A', 'T>C', 'T>G']
# Approximate COSMIC SBS1 signature
sbs1 = np.array([60, 12, 3, 2, 4, 12, 7], dtype=float)
sbs1 /= sbs1.sum()
# Random expectation (flat)
random_exp = np.ones(7) / 7

x = np.arange(len(mut_types))
ax.bar(x - 0.2, sbs1, 0.35, color='steelblue', alpha=0.8, label='SBS1 (CpG deamination)')
ax.bar(x + 0.2, random_exp, 0.35, color='lightgray', edgecolor='gray', label='Random expectation')
ax.set_xticks(x); ax.set_xticklabels(mut_types, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Fraction of mutations')
ax.set_title('Mutation signature SBS1:\nCpG transitions dominate')
ax.legend(fontsize=8)

# Panel 2: Mutation-selection balance
ax = axes[1]
s_range = np.logspace(-3, 0, 200)
q_add = mu / s_range
q_rec = np.sqrt(mu / s_range)
q_add = np.clip(q_add, 0, 1)
q_rec = np.clip(q_rec, 0, 1)

ax.loglog(s_range, q_add, color='steelblue', lw=2, label='Additive (q* = μ/s)')
ax.loglog(s_range, q_rec, color='tomato', lw=2, label='Recessive (q* = √(μ/s))')
ax.set_xlabel('Selection coefficient s')
ax.set_ylabel('Equilibrium allele frequency q*')
ax.set_title('Mutation-selection balance:\nrecessive alleles persist at higher frequency')
ax.legend()
ax.axhline(1e-4, color='gray', ls=':', lw=0.8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `titv_ratio(ref_alleles, alt_alleles)`. Given a VCF file with only SNPs,
   why would Ti/Tv < 1.8 suggest genotyping errors rather than biology?
2. Lynch syndrome is caused by loss of mismatch repair (MMR). What signature of mutations
   would you expect in tumours from Lynch syndrome patients (which mutation types,
   which contexts)?
3. A de novo mutation rate of 60 per generation means ~30 new mutations from the
   father and ~30 from the mother on average. If the paternal mutation rate increases
   by 1–2 mutations per year of age, how many more mutations does a child have if
   the father is 50 vs. 25 years old?
4. Implement `mutation_selection_equilibrium(mu, s, dominance)`. A recessive disease
   allele has s=0.5 in homozygotes and μ=10⁻⁵. What is the equilibrium carrier frequency?

---
## Quiz — Active Recall

1. What is a transition? What is a transversion? Which is more common and why?
2. Why are CpG sites mutation hotspots?
3. What is the Ti/Tv ratio? What would an unusually low ratio suggest?
4. What is the difference between germline and somatic mutations?
5. Which DNA repair pathway is mutated in BRCA1/2 carriers?

---
## Reflection

**Date completed:** ____________________

> *[A whole-exome sequencing report shows Ti/Tv = 4.1 in coding variants. Is this
> normal, low, or high? What does it suggest about the biology or the pipeline?]*

---
*Next: `04_genetic_drift_selection_revisited.ipynb`*